# Test4Py setup pilot on Kaggle
Enable **Internet** and **GPU** in Notebook Settings. Run cells in order.
The notebook kernel may use Python 3.13; the pilot runs in isolated Python 3.11.
The environment and model cache use `/tmp`; retain reports in `/kaggle/working`.
Save & Run All starts a separate session and must execute setup again.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone

SOURCE = Path('/kaggle/working/Test4Py-Setup-pilot-temp')
REPOSITORY = 'https://github.com/nguyendevwk/Test4Py-Setup-pilot-temp.git'
# Set to a recorded source commit for a source-pinned replay.
SOURCE_COMMIT = None
if not SOURCE.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(SOURCE)], check=True)
if SOURCE_COMMIT:
    subprocess.run(['git', '-C', str(SOURCE), 'fetch', 'origin', SOURCE_COMMIT], check=True)
    subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(SOURCE)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
print('Kernel:', sys.version)
os.environ['HF_HOME'] = '/tmp/test4py-hf-cache'
os.environ['UV_CACHE_DIR'] = '/tmp/test4py-uv-cache'
os.environ['UV_PYTHON_INSTALL_DIR'] = '/tmp/test4py-python'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ENV_REPORTS = Path('/kaggle/working') / ('pilot-environment-' + RUN_ID)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.6.17'], check=True)
subprocess.run([sys.executable, 'bootstrap_kaggle.py', '--venv', '/tmp/test4py-venv',
                '--report-dir', str(ENV_REPORTS)], check=True)
PYTHON = '/tmp/test4py-venv/bin/python'
subprocess.run([PYTHON, '-c', 'import torch; print(torch.__version__, torch.version.cuda); '
                'assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings"; '
                'print(torch.cuda.get_device_name(0))'], check=True)


In [ ]:
VALIDATION = Path('/kaggle/working') / ('validate-real-' + RUN_ID)
subprocess.run([PYTHON, 'run.py', '--validate-only', '--output', str(VALIDATION)], check=True)


In [ ]:
OUTPUT = Path('/kaggle/working') / ('pilot-qwen-1b-' + RUN_ID)
# For replay, add --model-revision and --embedding-revision from experiment.json.
subprocess.run([PYTHON, 'run.py', '--mode', 'compare', '--rounds', '1',
                '--output', str(OUTPUT)], check=True)


In [ ]:
import json
print('Reports:', OUTPUT)
print(json.dumps(json.loads((OUTPUT / 'comparison.json').read_text()), indent=2, ensure_ascii=False))


Generated pytest: `<output>/<mode>/PyCG/Test4DT_tests/`.
Keep `comparison.json`, `experiment.json`, `environment.json` and each arm's logs.
A failed command is not a research score: inspect its reports before interpreting results.
Official platform guidance: https://www.kaggle.com/docs/notebooks
